In [ ]:
import sys
import os

# 1. DRIVE MOUNT AND PATHS SETUP
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory: {os.getcwd()}")
except ImportError:
    # Fallback for local execution
    DRIVE_ROOT = '.'
    print("Local execution")

In [ ]:
!pip install pm4py

**EXPORT LOGS**

In [ ]:
"""
Convert the per-video CSV logs written by the action recognition script into a log with
non-overlapping action intervals, one row per action and one object per row for each object type.

Input  (one row per sliding window and per person, objects marked 1 = associated to the person):
    Timestamp, person_1000, person_1001, plate_1000, plate_1001, cup_1001, ..., Action
    0.00s-2.00s, 1, 0, 1, 1, 0, ..., Reaching; CarryingWhileLocomoting

Output (one row per action interval and, when needed, per object):
    Start, End, Person, Cup, Plate, ..., Action
    0.00, 2.50, person_1000, cup_1001, plate_1000, Reaching
    0.00, 2.50, person_1000,          , plate_1001, Reaching

- Start / End (seconds): every window "owns" the time between its midpoint and the midpoints of
  the previous / next window (never outside the window itself), so the time owned by different
  windows never overlaps. Successive windows of the same person with the same action are merged
  into one row going from the start of the first to the end of the last. Windows separated by a
  time gap (the person was not analyzed there) are not merged.
- Parallel actions ("A; B") go on separate rows. Rows of different actions can share the same
  time span, because the actions are parallel.
- Object columns: one per object type (the type is the id without its numeric suffix, so
  plate_1000 and plate_1001 go in "Plate"). People are objects too: the analyzed person is in the
  "Person" column of every row.
- Objects of an interval: the ids associated to the person in any of the merged windows. If
  there are several ids of the same type they go on separate rows (same Start, End and Action),
  one id per row in each type column.
"""
import os
import re
import csv
import glob

# PATHS
PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
PATH_LOGS = os.path.join(PATH_DATA_ROOT, "action_recognition_logs_temp")
PATH_LOGS_CONVERTED = os.path.join(PATH_DATA_ROOT, "action_recognition_logs_converted_temp")

PERSON_SEPARATOR = ", "

_TIMESTAMP_RE = re.compile(r"^\s*([\d.]+)s?\s*-\s*([\d.]+)s?\s*$")
_ID_SUFFIX_RE = re.compile(r"[_\-\s]*\d+$")
_EPS = 1e-9


def object_type(object_id):
    """'plate_1000' -> 'Plate', 'cutting_board_7' -> 'Cutting board', 'person_1000' -> 'Person'."""
    name = _ID_SUFFIX_RE.sub("", object_id).replace("_", " ").strip() or object_id
    return name[:1].upper() + name[1:]


def parse_window(timestamp):
    """'0.00s-2.00s' -> (0.0, 2.0) seconds."""
    match = _TIMESTAMP_RE.match(timestamp)
    if not match:
        raise ValueError(f"Unexpected timestamp format: {timestamp!r}")
    return float(match.group(1)), float(match.group(2))


def convert_log(in_path, out_path):
    """Convert one log file. Returns (window rows read, action rows written)."""
    with open(in_path, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    if not rows:
        return 0, 0

    header = rows[0]
    if len(header) < 3 or header[0] != "Timestamp" or header[-1] != "Action":
        raise ValueError(f"{in_path}: unexpected header {header}")
    object_ids = header[1:-1]
    # People first, then the other types in alphabetical order
    types = sorted({object_type(o) for o in object_ids}, key=lambda t: (t != "Person", t.lower()))

    # One entry per (window, person). If a video was re-run, the same window can appear more than
    # once in the log (rows are appended): the last one is kept.
    latest = {}
    n_read = 0
    for row in rows[1:]:
        if not row or all(not cell.strip() for cell in row):
            continue
        n_read += 1
        if len(row) != len(header):
            print(f"  [Warning] {os.path.basename(in_path)}: row with {len(row)} columns instead of {len(header)}, skipped")
            continue
        timestamp, flags, action = row[0], row[1:-1], row[-1]
        present = [oid for oid, flag in zip(object_ids, flags) if flag.strip() == "1"]
        persons = tuple(sorted(o for o in present if object_type(o) == "Person"))
        latest[(timestamp, persons)] = (present, action)

    windows_by_person = {}
    for (timestamp, persons), (present, action) in latest.items():
        start, end = parse_window(timestamp)
        windows_by_person.setdefault(persons, []).append({
            "start": start, "end": end, "center": (start + end) / 2,
            "objects": [o for o in present if object_type(o) != "Person"],
            "actions": list(dict.fromkeys(a.strip() for a in action.split(";") if a.strip())) or [""],
        })

    # Merge successive windows with the same action into intervals (one list of runs per person)
    runs = []
    for persons, windows in windows_by_person.items():
        windows.sort(key=lambda w: w["center"])
        for i, w in enumerate(windows):
            # time owned by this window: up to the midpoint with the neighbours, never outside the window
            w["seg_start"] = w["start"]
            w["seg_end"] = w["end"]
            if i > 0:
                w["seg_start"] = max(w["start"], (windows[i - 1]["center"] + w["center"]) / 2)
            if i + 1 < len(windows):
                w["seg_end"] = min(w["end"], (w["center"] + windows[i + 1]["center"]) / 2)

        open_runs = {}
        for w in windows:
            new_open = {}
            for action in w["actions"]:
                run = open_runs.get(action)
                if run is not None and abs(run["end"] - w["seg_start"]) < _EPS:
                    run["end"] = w["seg_end"]          # same action in the next, contiguous window
                    run["objects"].update(w["objects"])
                else:
                    run = {"start": w["seg_start"], "end": w["seg_end"], "persons": persons,
                           "action": action, "objects": set(w["objects"])}
                    runs.append(run)
                new_open[action] = run
            open_runs = new_open

    runs.sort(key=lambda r: (r["start"], r["persons"]))  # stable: action order inside a window is kept

    out_rows = []
    for run in runs:
        ids_by_type = {t: [] for t in types}
        for oid in sorted(run["objects"]):
            ids_by_type[object_type(oid)].append(oid)
        person_cell = PERSON_SEPARATOR.join(run["persons"])
        n_rows = max([1] + [len(ids) for t, ids in ids_by_type.items() if t != "Person"])
        for k in range(n_rows):  # several objects of the same type -> several rows
            cells = [person_cell if t == "Person" else (ids_by_type[t][k] if k < len(ids_by_type[t]) else "")
                     for t in types]
            out_rows.append([f"{run['start']:.2f}", f"{run['end']:.2f}"] + cells + [run["action"]])

    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Start", "End"] + types + ["Action"])
        writer.writerows(out_rows)
    return n_read, len(out_rows)


def convert_directory(in_dir, out_dir):
    log_files = sorted(glob.glob(os.path.join(in_dir, "*_log.csv")))
    if not log_files:
        print(f"No *_log.csv file found in {in_dir}")
        return 0
    for path in log_files:
        out_path = os.path.join(out_dir, os.path.basename(path))
        n_in, n_out = convert_log(path, out_path)
        print(f"[+] {os.path.basename(path)}: {n_in} window rows -> {n_out} action rows")
    print(f"Converted logs saved in {out_dir}")
    return len(log_files)


if __name__ == "__main__":
    convert_directory(PATH_LOGS, PATH_LOGS_CONVERTED)

**GENERATE OCEL**

In [ ]:
"""
Convert the action-interval logs (output of the log conversion script) into a single OCEL 2.0 JSON file.

Input  (one CSV per clip, one row per action interval and per object):
    Start, End, Person, Cup, Plate, ..., Action
    3.50, 6.50, person_1000, cup_1002, plate_1001, CarryingWhileLocomoting

Output: one OCEL 2.0 JSON (objectTypes, eventTypes, objects, events) for all the clips.

Mapping:
- Event: one per (Start, End, Person, Action). The rows that only differ in the objects (several objects of
  the same type are written on separate rows) are merged back into one event.
- Event type: the Action. Rows with an empty action (window with no prediction) are not events.
- Event time: Start. Synthetic timestamp = REFERENCE_DATETIME + offset of the part + Start. Only order and
  durations are meaningful, not the date or the time of day. The End goes into the event attribute "end".
- Object type: the name of the column (Person, Cup, Plate, "Cutting board", ...).
- Object id: "<clip>:<id>", e.g. "cam0_15_scene1_part5:person_1000". Ids restart in every clip, so they are
  prefixed with the clip name to avoid collisions. The same physical cup in two clips is therefore two objects.
- Event-to-object relations: PERSON_QUALIFIER for the person, OBJECT_QUALIFIER for every other object.
- Event id: "<clip>:e<k>", k = order of the event inside the clip file.

"""
import os
import re
import csv
import glob
import json
import argparse
from datetime import datetime, timedelta, timezone

# PATHS
PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
PATH_LOGS_CONVERTED = os.path.join(PATH_DATA_ROOT, "action_recognition_logs_converted_temp")
PATH_OCEL = os.path.join(PATH_DATA_ROOT, "action_recognition_logs_ocel_temp")
OUTPUT_FILE = "ocel2.json"
LOG_GLOB = "*_log.csv"
VERIFY= True

# SYNTHETIC TIME
# Arbitrary reference: the logs are offline annotations, the real recording date-time is unknown.
REFERENCE_DATETIME = datetime(2000, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
PIPELINE_FPS = 30
PART_FRAMES = 300   # a TUM Kitchen part has at most 300 frames (10 s): the offset of part Z is (Z-1) * 10 s
# Clip names like cam0_15_scene1_part5 -> part 5. Clips without this suffix (e.g. PennAction videos) have offset 0.
PART_RE = re.compile(r"_scene(\d+)_part(\d+)$", re.IGNORECASE)

# LOG FORMAT
PERSON_COLUMN = "Person"
PERSON_SEPARATOR = ", "   # same separator used by the log conversion script when a row has several persons
SKIP_EMPTY_ACTION = True

# OCEL
PERSON_QUALIFIER = "actor"
OBJECT_QUALIFIER = "associated-object"   # the association step says "associated", not "used"
EVENT_ATTRIBUTES = [("end", "time"), ("clip", "string")]


def clip_name(path):
    """'.../cam0_15_scene1_part5_log.csv' -> 'cam0_15_scene1_part5'."""
    name = os.path.splitext(os.path.basename(path))[0]
    return name[:-len("_log")] if name.endswith("_log") else name


def part_offset_seconds(clip):
    match = PART_RE.search(clip)
    if not match:
        return 0.0
    scene, part = int(match.group(1)), int(match.group(2))
    if scene != 1:
        print(f"  [Warning] {clip}: scene {scene}, the part offset assumes a single scene per video.")
    return (part - 1) * PART_FRAMES / PIPELINE_FPS


def to_iso(seconds):
    """Seconds from the start of the recording -> ISO 8601 UTC timestamp with milliseconds."""
    t = REFERENCE_DATETIME + timedelta(milliseconds=round(seconds * 1000))
    return t.strftime("%Y-%m-%dT%H:%M:%S.") + f"{t.microsecond // 1000:03d}Z"


def read_clip(path):
    """Read one converted log. Returns (events, rows read, rows skipped).
    Rows with the same (Start, End, Person, Action) are merged into one event (union of the objects)."""
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    if not rows:
        return [], 0, 0

    header = [h.strip() for h in rows[0]]
    if len(header) < 3 or header[:2] != ["Start", "End"] or header[-1] != "Action":
        raise ValueError(f"{path}: unexpected header {header}")
    type_columns = header[2:-1]

    events = {}
    n_read = n_skipped = 0
    for row in rows[1:]:
        if not row or all(not cell.strip() for cell in row):
            continue
        n_read += 1
        if len(row) != len(header):
            print(f"  [Warning] {os.path.basename(path)}: row with {len(row)} columns instead of {len(header)}, skipped")
            n_skipped += 1
            continue
        start, end, action = row[0].strip(), row[1].strip(), row[-1].strip()
        if not action and SKIP_EMPTY_ACTION:
            n_skipped += 1
            continue

        persons_cell, objects = "", []
        for column, cell in zip(type_columns, row[2:-1]):
            cell = cell.strip()
            if not cell:
                continue
            if column == PERSON_COLUMN:
                persons_cell = cell
            else:
                objects.append((column, cell))

        event = events.setdefault((start, end, persons_cell, action), {
            "start": float(start), "end": float(end), "action": action,
            "persons": [p.strip() for p in persons_cell.split(PERSON_SEPARATOR) if p.strip()],
            "objects": set(),
        })
        event["objects"].update(objects)
    return list(events.values()), n_read, n_skipped


def build_ocel(paths):
    object_types, event_types = set(), set()
    objects = {}   # global object id -> type
    events = []

    for path in paths:
        clip = clip_name(path)
        offset = part_offset_seconds(clip)
        clip_events, n_read, n_skipped = read_clip(path)

        for k, ev in enumerate(clip_events, 1):
            relationships = []
            for person in ev["persons"]:
                oid = f"{clip}:{person}"
                objects[oid] = PERSON_COLUMN
                relationships.append({"objectId": oid, "qualifier": PERSON_QUALIFIER})
            for obj_type, obj_id in sorted(ev["objects"]):
                oid = f"{clip}:{obj_id}"
                objects[oid] = obj_type
                relationships.append({"objectId": oid, "qualifier": OBJECT_QUALIFIER})

            object_types.update(objects[r["objectId"]] for r in relationships)
            event_types.add(ev["action"])
            events.append({
                "id": f"{clip}:e{k}",
                "type": ev["action"],
                "time": to_iso(offset + ev["start"]),
                "attributes": [
                    {"name": "end", "value": to_iso(offset + ev["end"])},
                    {"name": "clip", "value": clip},
                ],
                "relationships": relationships,
            })
        print(f"[+] {os.path.basename(path)}: {n_read} rows -> {len(clip_events)} events "
              f"({n_skipped} rows skipped), offset {offset:.1f}s")

    events.sort(key=lambda e: e["time"])   # stable: the order inside a clip is kept for equal times
    return {
        "objectTypes": [{"name": t, "attributes": []} for t in sorted(object_types)],
        "eventTypes": [{"name": t, "attributes": [{"name": n, "type": ty} for n, ty in EVENT_ATTRIBUTES]}
                       for t in sorted(event_types)],
        "objects": [{"id": oid, "type": t, "attributes": [], "relationships": []}
                    for oid, t in sorted(objects.items())],
        "events": events,
    }


def verify(path):
    """Import the file with pm4py and flatten it on the Person type (= one case per clip and actor)."""
    try:
        import pm4py
    except ImportError:
        print("pm4py not installed: verification skipped (pip install pm4py).")
        return
    ocel = pm4py.read_ocel2_json(path)
    print("\n[pm4py] import OK")
    print(ocel)
    print("[pm4py] object types:", pm4py.ocel_get_object_types(ocel))
    try:
        flat = pm4py.ocel_flattening(ocel, PERSON_COLUMN)
        print(f"[pm4py] flattening on {PERSON_COLUMN}: {len(flat)} events, "
              f"{flat['case:concept:name'].nunique()} cases")
    except Exception as exc:   # flattening is only an extra check
        print(f"[pm4py] flattening failed: {exc!r}")


if __name__ == "__main__":
    out = os.path.join(PATH_OCEL, OUTPUT_FILE)

    log_files = sorted(glob.glob(os.path.join(PATH_LOGS_CONVERTED, LOG_GLOB)))
    if not log_files:
        raise SystemExit(f"No file in {PATH_LOGS_CONVERTED}")

    ocel = build_ocel(log_files)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(ocel, f, indent=2, ensure_ascii=False)
    print(f"\n{len(log_files)} clips -> {len(ocel['events'])} events, {len(ocel['objects'])} objects, "
          f"{len(ocel['eventTypes'])} event types, {len(ocel['objectTypes'])} object types")
    print(f"OCEL 2.0 JSON saved in {out}")

    if VERIFY:
        verify(out)

**RECONCILE OBJECT IDS**

In [ ]:
"""Reconcile per-clip object IDs in an exported OCEL 2.0 JSON event log.

This is a post-processing step. It uses only the bounding boxes in the
Object Detection JSONs, independently of any manually annotated ground truth.
It matches same-class tracks between directly consecutive _partN clips,
then replaces object IDs in OCEL objects and event relationships. Uncertain
matches remain distinct (conservative behavior).

Dependencies: numpy, scipy.

"""
import argparse
import csv
import json
import math
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy.optimize import linear_sum_assignment

CLIP_RE = re.compile(r"^(?P<video>.+)_scene(?P<scene>\d+)_part(?P<part>\d+)$")


def box_at_edge(data, edge, max_gap_frames):
    boxes = data.get('bbox', [])
    if not boxes:
        return None
    inds = (range(min(max_gap_frames, len(boxes))) if edge == 'start'
            else range(len(boxes)-1, max(-1, len(boxes)-max_gap_frames-1), -1))
    for index in inds:
        box = boxes[index]
        if not isinstance(box, (list, tuple)) or len(box) != 4:
            continue
        try:
            x1, y1, x2, y2 = map(float, box)
        except (ValueError, TypeError):
            continue
        if (all(math.isfinite(z) for z in (x1,y1,x2,y2))
                and 0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1):
            return (x1, y1, x2, y2)
    return None


def tracks_at_boundary(tracks, edge, gap):
    out = []
    for local_key, data in tracks.items():
        b = box_at_edge(data, edge, gap)
        if b is not None:
            cls = str(data.get('class_name', local_key.rsplit('_', 1)[0])).strip().lower()
            out.append(dict(id=local_key, cls=cls, bbox=b))
    return out


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(ix2-ix1, 0) * max(iy2-iy1, 0)
    aa = (a[2]-a[0])*(a[3]-a[1])
    bb = (b[2]-b[0])*(b[3]-b[1])
    return intersection / (aa+bb-intersection) if (aa+bb-intersection) else 0.0


def measurements(a,b):
    x = iou(a,b)
    cx1,cy1=(a[0]+a[2])/2,(a[1]+a[3])/2
    cx2,cy2=(b[0]+b[2])/2,(b[1]+b[3])/2
    dist=math.hypot(cx1-cx2,cy1-cy2)
    area_a=(a[2]-a[0])*(a[3]-a[1]); area_b=(b[2]-b[0])*(b[3]-b[1])
    size_ratio=min(area_a,area_b)/max(area_a,area_b)
    return x,dist,size_ratio


def find_pairs(prev_tracks, next_tracks, gap, min_iou, max_center_dist, ambiguity_margin):
    left = tracks_at_boundary(prev_tracks, 'end', gap)
    right = tracks_at_boundary(next_tracks, 'start', gap)
    if not left or not right:
        return [], len(left), len(right)
    costs=np.full((len(left),len(right)), 1e6, dtype=float)
    scores=np.full_like(costs, -1.0)
    stats={}
    for i,a in enumerate(left):
        for j,b in enumerate(right):
            if a['cls'] != b['cls']:
                continue
            ov,dist,sz=measurements(a['bbox'],b['bbox'])
            # Center-only matches require comparable object size; otherwise keep separate.
            eligible=(ov>=min_iou) or (dist<=max_center_dist and sz>=0.65)
            if not eligible:
                continue
            score=0.75*ov+0.20*max(0.0,1.0-dist/0.15)+0.05*sz
            scores[i,j]=score
            costs[i,j]=1.0-score
            stats[(i,j)]=(ov,dist,sz,score)
    if not stats:
        return [],len(left),len(right)
    rows,cols=linear_sum_assignment(costs)
    found=[]
    for i,j in zip(rows,cols):
        if (i,j) not in stats:
            continue
        ov,dist,sz,score=stats[(i,j)]
        alt_left=sorted((scores[i,k] for k in range(len(right)) if k!=j and scores[i,k]>=0),reverse=True)
        alt_right=sorted((scores[k,j] for k in range(len(left)) if k!=i and scores[k,j]>=0),reverse=True)
        ambiguous=(alt_left and score-alt_left[0] < ambiguity_margin) or (alt_right and score-alt_right[0] < ambiguity_margin)
        found.append(dict(prev_id=left[i]['id'], next_id=right[j]['id'],
                          cls=left[i]['cls'], iou=round(ov,4),
                          center_distance=round(dist,4), size_ratio=round(sz,4),
                          score=round(score,4), status='ambiguous' if ambiguous else 'matched'))
    return found,len(left),len(right)


class UnionFind:
    def __init__(self):
        self.parent={}
    def root(self,item):
        self.parent.setdefault(item,item)
        if self.parent[item]!=item:
            self.parent[item]=self.root(self.parent[item])
        return self.parent[item]
    def union(self,a,b):
        ra,rb=self.root(a),self.root(b)
        if ra!=rb:
            self.parent[rb]=ra  # Prefer the earliest clip identifier.


def run(args):
    root=Path(args.tracking_root)
    groups=defaultdict(list)
    for folder in root.iterdir():
        if not folder.is_dir():
            continue
        match=CLIP_RE.fullmatch(folder.name)
        if not match:
            continue
        p=folder/(folder.name+'.json')
        if not p.exists():
            continue
        with p.open('r',encoding='utf8') as f:
            tracks=json.load(f).get('detected_objects',{})
        groups[(match['video'],int(match['scene']))].append((int(match['part']),folder.name,tracks))

    dsu=UnionFind()
    report=[]
    for _,entries in sorted(groups.items()):
        entries.sort(key=lambda t:t[0])
        for (part1,name1,tracks1),(part2,name2,tracks2) in zip(entries,entries[1:]):
            if part2!=part1+1:
                continue
            found,n1,n2=find_pairs(tracks1,tracks2,args.max_gap_frames,
                                  args.min_iou,args.max_center_dist,args.ambiguity_margin)
            if not found:
                report.append(dict(clip_previous=name1,clip_next=name2,prev_id='',next_id='',
                                   cls='',iou='',center_distance='',size_ratio='',score='',
                                   status=f'no candidates (end={n1}, start={n2})'))
            for m in found:
                report.append(dict(clip_previous=name1,clip_next=name2,**m))
                if m['status']=='matched':
                    dsu.union(f"{name1}:{m['prev_id']}",f"{name2}:{m['next_id']}")

    with open(args.ocel_in,encoding='utf8') as f:
        ocel=json.load(f)
    objects=ocel.get('objects',[])
    aliases={}
    # A track might be absent from the OCEL because it was never associated
    # with an event. Choose an OCEL-present representative in that situation.
    # Group by OCEL object type as well, never merging different typed objects.
    representatives={}
    for obj in objects:
        group=(dsu.root(obj['id']),obj.get('type'))
        representatives.setdefault(group,obj['id'])
        aliases[obj['id']]=representatives[group]

    uniq={}
    for obj in objects:
        new_id=aliases[obj['id']]
        if new_id not in uniq:
            copy=dict(obj)
            copy['id']=new_id
            uniq[new_id]=copy
    ocel['objects']=list(uniq.values())
    changed_relations=0
    for event in ocel.get('events',[]):
        rels=[]; seen=set()
        for rel in event.get('relationships',[]):
            item=dict(rel)
            old=item.get('objectId')
            new=aliases.get(old,old)
            if new!=old:
                changed_relations+=1
            item['objectId']=new
            key=(item.get('objectId'),item.get('qualifier'))
            if key not in seen:
                rels.append(item);seen.add(key)
        event['relationships']=rels

    Path(args.ocel_out).parent.mkdir(parents=True,exist_ok=True)
    with open(args.ocel_out,'w',encoding='utf8') as f:
        json.dump(ocel,f,ensure_ascii=False,indent=2)
    Path(args.report).parent.mkdir(parents=True,exist_ok=True)
    cols=['clip_previous','clip_next','prev_id','next_id','cls','iou','center_distance',
          'size_ratio','score','status']
    with open(args.report,'w',newline='',encoding='utf8') as f:
        writer=csv.DictWriter(f,fieldnames=cols);writer.writeheader();writer.writerows(report)
    successes=sum(r['status']=='matched' for r in report)
    ambiguous=sum(r['status']=='ambiguous' for r in report)
    print(f"Boundary matches: {successes}; ambiguous and NOT merged: {ambiguous}")
    print(f"OCEL objects: {len(objects)} -> {len(ocel['objects'])}")
    print(f"Updated event-object relationships: {changed_relations}")
    print(f"Output: {args.ocel_out}\nAudit report: {args.report}")
    print("Note: inspect the report and boundary frames; this is not a validated re-identification system.")


if __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__,formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--tracking-root',required=True,help='Folder containing one subfolder per clip, each with <clip>.json')
    parser.add_argument('--ocel-in',required=True,help='Existing OCEL 2.0 JSON created by the pipeline')
    parser.add_argument('--ocel-out',required=True,help='Output OCEL 2.0 JSON with reconciled IDs')
    parser.add_argument('--report',default='id_matches.csv',help='CSV report of boundary matches')
    parser.add_argument('--max-gap-frames',type=int,default=10,help='Look at up to N first/last frames of a clip')
    parser.add_argument('--min-iou',type=float,default=0.30,help='Min IoU for a shape match; else require center + similar size')
    parser.add_argument('--max-center-dist',type=float,default=0.04,help='Max normalized center distance for a center-only match')
    parser.add_argument('--ambiguity-margin',type=float,default=0.08,help='Reject assignment if competing score is too close')
    run(parser.parse_args())


**MODEL DISCOVERY**

In [ ]:
"""
Process discovery on TUM Kitchen: Directly-Follows Graph, Inductive Miner and Heuristics Miner,
applied to (a) the log produced by the pipeline (ocel2.json) and (b) the log built from the
ground-truth labels (TUMKitchenGT/labels*.csv), to compare the two.

Steps:
  1. pipeline log: read ocel2.json (plain json)
  2. ground-truth log: read labels*.csv (25 Hz, left/right hand)
  3. for every source x level (L0/L1) x with/without CarryingWhileLocomoting:
       - DFG (full and filtered);
       - Inductive Miner at several noise thresholds
       - Heuristics Miner at several dependency thresholds
       - fitness, precision, F-score, simplicity of every model (token-based);
  4. comparison pipeline vs ground truth (when the GT is used):
       - conformance checking
       - direct comparison of the DFGs
       - time perspective
  5. save model_metrics.csv and log_statistics.csv.

Notes:
  - explicit mapping of 19_scene1 (UNANNOTATED) and 20_scene1 (labels_19.csv),
    consistent with ar_validation_tumkitchen.py
  - export the complete mapping to check;
  - export a supplemental interval-aware comparison and overlap statistics
"""
import os
import re
import glob
import json
from collections import Counter
import warnings
import numpy as np
import pandas as pd
import pm4py

warnings.filterwarnings("ignore")
try:   # silence the progress bars of the token-based replay
    from pm4py.util import constants
    constants.SHOW_PROGRESS_BAR = False
except Exception:
    pass

# ----------------------------------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------------------------------
PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
OCEL_FILE = os.path.join(PATH_DATA_ROOT, "action_recognition_logs_ocel", "ocel2.json")
GT_DIR = os.path.join(PATH_DATA_ROOT, "TUMKitchenGT")
OUT_DIR = os.path.join(PATH_DATA_ROOT, "process_discovery_aligned")

# ----------------------------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------------------------
CAMERA = "cam1"            # "cam0".."cam3" (prefix of the clip name), or None = all the cameras together
CASE_NOTION = "recording"  # "recording": one case per recording (all the parts of an episode, joined)
                           # "clip":      one case per (clip, actor), 10-second cases (pipeline log only)
KEEP_MAIN_PERSON_ONLY = True   # TUM has a single subject: in every clip keep only the track with the
                               # largest temporal coverage (the others are spurious detections)
LEVELS = ["L1", "L0"]      # L1 = Opening/Closing Door+Drawer merged into OpeningAContainer / ClosingAContainer
DROP_CARRYING = [False, True]   # run with and without the catch-all class CarryingWhileLocomoting
MERGE_REPEATS = False      # True: collapse consecutive identical activities (A, A -> A) within a case
MERGE_CLIP_BOUNDARIES = True  # Controlled sensitivity analysis: merge same L0 action across adjacent clips
BOUNDARY_TOLERANCE_S = 0.1    # Maximum gap/overlap (seconds) between adjacent clip fragments

IM_NOISE_THRESHOLDS = [0.0, 0.2, 0.4]   # Inductive Miner infrequent: 0 = no filtering
HM_DEPENDENCY_THRESHOLDS = [0.5, 0.8, 0.9]   # Heuristics Miner: higher = fewer, more reliable dependencies
DFG_MIN_EDGE_FRACTION = 0.10   # filtered DFG: keep the edges with count >= 10% of the most frequent one

# Ground truth
USE_GT = True
GT_FILE_GLOB = "labels*.csv"   # labels.csv (video 1), labels_2.csv (video 2), ... labels_19.csv (video 20)
GT_FPS = 25
# Same mapping used by ar_validation_tumkitchen.py.
# IMPORTANT: video index 19 (19_scene1) has NO labels.
# Video index 20 (20_scene1) is annotated in labels_19.csv.
# All other video indices map directly to labels_<index>.csv (1 -> labels.csv).
VIDEO_TO_LABEL_INDEX = {19: -1, 20: 19}
# startframe in the original 25-Hz TUM Kitchen video. The label CSV instance
# column is RELATIVE to this offset, as in the action-recognition validation.
GT_ROW_OFFSET = {
    1: 240, 2: 255, 3: 234, 4: 306, 5: 256, 6: 281,
    7: 240, 8: 237, 9: 254, 10: 215, 11: 290, 12: 265,
    13: 309, 14: 195, 15: 228, 16: 255, 17: 242, 18: 205,
    19: 200,
}
MIN_GT_RUN_FRAMES = 1          # ignore ground-truth runs shorter than this number of frames
RESTRICT_PIPELINE_TO_GT_EPISODES = True   # compare the two logs on the same episodes
RESTRICT_PIPELINE_TO_GT_SPAN = True       # the GT covers only a segment of each video: keep the pipeline
                                          # events that overlap that segment, so both logs cover the same time
TOP_K_EDGES = 10           # top-k most frequent directly-follows edges, for the overlap between the two DFGs
PER_TRACE_CONFORMANCE = True   # fitness of every pipeline trace on the GT models

COMPUTE_METRICS = True
SHOW_IN_NOTEBOOK = True    # display the PNGs in the Colab output

CARRYING = "CarryingWhileLocomoting"
CLASSES = ["OpeningADoor", "OpeningADrawer", "ClosingADoor", "ClosingADrawer", "Reaching",
           "TakingSomething", "LoweringAnObject", "ReleasingGraspOfSomething", CARRYING]
L1_MAP = {
    "OpeningADoor": "OpeningAContainer", "OpeningADrawer": "OpeningAContainer",
    "ClosingADoor": "ClosingAContainer", "ClosingADrawer": "ClosingAContainer",
}
REFERENCE = pd.Timestamp("2000-01-01", tz="UTC")   # same synthetic origin of the OCEL export script


# ----------------------------------------------------------------------------------------------
# PIPELINE LOG
# ----------------------------------------------------------------------------------------------
def episode_key(recording):
    """'cam1_15_scene1' -> '15_scene1'; 'gt_15_scene1' -> '15_scene1'."""
    return re.sub(r"^(cam\d+|gt)_", "", recording)


def episode_order(key):
    return tuple(int(x) for x in re.findall(r"\d+", key))


def load_event_table(path):
    """OCEL 2.0 JSON -> DataFrame with one row per (event, actor)."""
    with open(path, encoding="utf-8") as f:
        ocel = json.load(f)
    rows = []
    for ev in ocel["events"]:
        attrs = {a["name"]: a["value"] for a in ev.get("attributes", [])}
        for rel in ev.get("relationships", []):
            if rel["qualifier"] == "actor":
                rows.append({
                    "event_id": ev["id"], "action": ev["type"], "start": ev["time"],
                    "end": attrs.get("end", ev["time"]), "clip": attrs["clip"], "actor": rel["objectId"],
                })
    df = pd.DataFrame(rows)
    df["start"] = pd.to_datetime(df["start"], utc=True)
    df["end"] = pd.to_datetime(df["end"], utc=True)
    df["camera"] = df["clip"].str.split("_").str[0]
    df["recording"] = df["clip"].str.replace(r"_part\d+$", "", regex=True)
    df["episode"] = df["recording"].map(episode_key)
    return df


def select_main_person(df):
    """In every clip keep the actor with the largest total duration of actions."""
    df = df.assign(dur=(df["end"] - df["start"]).dt.total_seconds())
    main = df.groupby(["clip", "actor"])["dur"].sum().reset_index().sort_values("dur", ascending=False) \
             .drop_duplicates("clip")[["clip", "actor"]]
    return df.merge(main, on=["clip", "actor"]).drop(columns="dur")


# ----------------------------------------------------------------------------------------------
# CROSS-CLIP EVENT FRAGMENTATION: controlled sensitivity analysis (pipeline only)
# ----------------------------------------------------------------------------------------------
def merge_cross_clip_fragments(events, tolerance_s=0.1):
    """Merge only same-L0-label fragments straddling consecutive clip boundaries.

    Works at the original action level (before L1 mapping) and never merges two
    fragments belonging to the same clip. It handles chains spanning >2 clips.
    Returns (merged event table, audit table of pairwise merges).
    """
    if tolerance_s < 0:
        raise ValueError("tolerance_s must be nonnegative")
    if events.empty:
        return events.copy(), pd.DataFrame()

    pattern = re.compile(r"_part(\d+)$")
    tol = pd.Timedelta(seconds=tolerance_s)
    output, audit = [], []
    # recording already contains camera; do not merge between separate videos.
    for (recording, action), group in events.groupby(["recording", "action"], sort=False):
        group = group.sort_values(["start", "end", "clip"])
        current = None
        last_clip = None
        last_end = None
        for _, ev in group.iterrows():
            ev = ev.copy()
            if current is not None:
                m1 = pattern.search(str(last_clip))
                m2 = pattern.search(str(ev["clip"]))
                gap = ev["start"] - last_end
                adjacent = bool(m1 and m2 and int(m2.group(1)) == int(m1.group(1)) + 1)
                # Only adjacent clip parts and very short temporal gap/overlap.
                if adjacent and -tol <= gap <= tol:
                    audit.append({
                        "recording": recording,
                        "action": action,
                        "clip_a": last_clip,
                        "clip_b": ev["clip"],
                        "end_a": last_end,
                        "start_b": ev["start"],
                        "gap_s": gap.total_seconds(),
                    })
                    current["end"] = max(current["end"], ev["end"])
                    last_clip = ev["clip"]
                    last_end = ev["end"]
                    continue
                output.append(current)
            current = ev
            last_clip = ev["clip"]
            last_end = ev["end"]
        if current is not None:
            output.append(current)

    merged = pd.DataFrame(output, columns=events.columns).reset_index(drop=True)
    audit_df = pd.DataFrame(audit, columns=[
        "recording", "action", "clip_a", "clip_b", "end_a", "start_b", "gap_s"
    ])
    assert len(merged) == len(events) - len(audit_df)
    return merged, audit_df


# ----------------------------------------------------------------------------------------------
# GROUND-TRUTH LOG
# ----------------------------------------------------------------------------------------------
def gt_file_index(path):
    """labels.csv -> 1, labels_2.csv -> 2, labels_10.csv -> 10."""
    m = re.search(r"_(\d+)$", os.path.splitext(os.path.basename(path))[0])
    return int(m.group(1)) if m else 1


def read_gt_file(path):
    """Returns (long table instance/label restricted to CLASSES, number of frames, last instance,
    unknown labels, first instance)."""
    g = pd.read_csv(path)
    g.columns = [c.strip().lower() for c in g.columns]
    g = g.drop_duplicates("instance").sort_values("instance")
    label_index = gt_file_index(path)
    if label_index not in GT_ROW_OFFSET:
        raise ValueError(f"No GT_ROW_OFFSET for {os.path.basename(path)}")
    g["instance"] = pd.to_numeric(g["instance"], errors="raise").astype("int64") + GT_ROW_OFFSET[label_index]
    long = pd.concat([g[["instance", "lefthand"]].rename(columns={"lefthand": "label"}),
                      g[["instance", "righthand"]].rename(columns={"righthand": "label"})])
    long = long.dropna().drop_duplicates()
    long["label"] = long["label"].astype(str).str.strip()
    unknown = long.loc[~long["label"].isin(CLASSES), "label"].value_counts()
    return long[long["label"].isin(CLASSES)], len(g), int(g["instance"].max()), unknown, int(g["instance"].min())


def gt_runs(long):
    """Maximal runs of consecutive frames in which a class is active -> (label, first frame, end frame)."""
    rows = []
    for label, grp in long.groupby("label"):
        inst = grp["instance"].sort_values().to_numpy()
        for seg in np.split(inst, np.where(np.diff(inst) != 1)[0] + 1):   # a gap in the GT breaks the run
            if len(seg) >= MIN_GT_RUN_FRAMES:
                rows.append((label, int(seg[0]), int(seg[-1]) + 1))
    return rows


def video_index_from_episode(episode):
    """'scene1' -> 1; '20_scene1' -> 20, matching the validation script."""
    match = re.fullmatch(r"(?:(\d+)_)?scene\d+", episode)
    if not match:
        raise ValueError(f"Unexpected TUM episode name: {episode!r}")
    return int(match.group(1)) if match.group(1) else 1


def map_gt_files(events_all, gt_data):
    """Explicit video -> label mapping, identical to the AR validation script.

    Do NOT zip episodes and GT files: video 19 has no annotations, while
    video 20 uses labels_19.csv.
    """
    episodes = sorted(events_all["episode"].unique(), key=episode_order)
    span = (events_all.groupby("episode")["end"].max() - REFERENCE).dt.total_seconds()
    gt_by_index = {gt_file_index(p): p for p in gt_data}
    mapping = {}
    skipped = []
    for episode in episodes:
        video_index = video_index_from_episode(episode)
        label_index = VIDEO_TO_LABEL_INDEX.get(video_index, video_index)
        if label_index == -1:
            skipped.append(episode)
            continue
        if label_index not in gt_by_index:
            raise ValueError(f"No labels_{label_index} for episode {episode}")
        mapping[episode] = gt_by_index[label_index]

    if len(set(mapping.values())) != len(mapping):
        raise ValueError("The mapping assigns one GT file to multiple episodes")
    if set(mapping.values()) != set(gt_data):
        unused = sorted(set(gt_data) - set(mapping.values()))
        raise ValueError(f"Some GT files were not matched with processed episodes: {unused}")

    print("\nEpisode <-> label file (GT frame offsets applied; end times in video seconds):")
    for episode, path in mapping.items():
        first, last = gt_data[path][4], gt_data[path][2]
        gt_begin, gt_end = first / GT_FPS, (last + 1) / GT_FPS
        flag = "  <-- CHECK: annotated interval exceeds processed video" if gt_end > span[episode] + 2 else ""
        print(f"   {episode:<14} {os.path.relpath(path, GT_DIR):<24} "
              f"GT [{gt_begin:6.1f}s, {gt_end:6.1f}s] "
              f"video {span[episode]:6.1f}s{flag}")
    if skipped:
        print(f"Skipped unannotated episodes: {skipped}")

    mapping_rows = []
    for episode, path in mapping.items():
        label_index = gt_file_index(path)
        first, last = gt_data[path][4], gt_data[path][2]
        mapping_rows.append({
            "episode": episode,
            "video_index": video_index_from_episode(episode),
            "label_index": label_index,
            "gt_file": os.path.basename(path),
            "gt_start_seconds": first / GT_FPS,
            "gt_end_seconds": (last + 1) / GT_FPS,
            "pipeline_last_event_seconds": float(span[episode]),
        })
    pd.DataFrame(mapping_rows).to_csv(os.path.join(OUT_DIR, "gt_episode_mapping.csv"), index=False)
    return mapping


def build_gt_events(mapping, gt_data):
    rows = []
    for episode, path in mapping.items():
        long = gt_data[path][0]
        for label, first, end in gt_runs(long):
            rows.append({
                "event_id": f"gt_{episode}:{label}:{first}", "action": label,
                "start": REFERENCE + pd.to_timedelta(first / GT_FPS, unit="s"),
                "end": REFERENCE + pd.to_timedelta(end / GT_FPS, unit="s"),
                "clip": f"gt_{episode}", "actor": f"gt_{episode}:person", "camera": "gt",
                "recording": f"gt_{episode}", "episode": episode,
            })
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------
# LOG PREPARATION
# ----------------------------------------------------------------------------------------------
def build_log(events, level, drop_carrying, notion):
    """Event table -> pm4py DataFrame (case:concept:name, concept:name, time:timestamp)."""
    df = events.copy()
    df["concept:name"] = df["action"]
    if level == "L1":
        df["concept:name"] = df["concept:name"].replace(L1_MAP)
    if drop_carrying:
        df = df[df["concept:name"] != CARRYING]
    df["case:concept:name"] = df["recording"] if notion == "recording" else df["actor"]

    df = df.sort_values(["case:concept:name", "start", "concept:name"]).reset_index(drop=True)
    if MERGE_REPEATS:
        same = (df["case:concept:name"] == df["case:concept:name"].shift()) & \
               (df["concept:name"] == df["concept:name"].shift())
        df = df[~same].reset_index(drop=True)
    df["time:timestamp"] = df["start"] + pd.to_timedelta(df.groupby("case:concept:name").cumcount(), unit="us")
    return df[["case:concept:name", "concept:name", "time:timestamp"]]


def log_stats(df):
    lens = df.groupby("case:concept:name").size()
    n_var = df.groupby("case:concept:name")["concept:name"].apply(tuple).nunique()
    return {"cases": lens.size, "events": len(df), "activities": df["concept:name"].nunique(),
            "variants": n_var, "trace_len_min": int(lens.min()), "trace_len_mean": round(float(lens.mean()), 1),
            "trace_len_max": int(lens.max())}


# ----------------------------------------------------------------------------------------------
# DISCOVERY
# ----------------------------------------------------------------------------------------------
def show(path):
    if SHOW_IN_NOTEBOOK:
        try:
            from IPython.display import Image, display
            print(os.path.basename(path))
            display(Image(path))
        except Exception:
            pass


def quality(df, net, im, fm):
    """Token-based fitness / precision / F-score / simplicity of a Petri net."""
    if not COMPUTE_METRICS:
        return {}
    try:
        fit = pm4py.fitness_token_based_replay(df, net, im, fm)
        prec = pm4py.precision_token_based_replay(df, net, im, fm)
        from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator
        f = fit["log_fitness"]
        return {
            "fitness": round(f, 4),
            "perc_fitting_traces": round(fit.get("percentage_of_fitting_traces", float("nan")), 2),
            "precision": round(prec, 4),
            "f_score": round(2 * f * prec / (f + prec), 4) if (f + prec) > 0 else 0.0,
            "simplicity": round(simplicity_evaluator.apply(net), 4),
        }
    except Exception as exc:
        print(f"  [metrics skipped] {exc!r}")
        return {}


def run_variant(df, tag):
    results = []
    # --- DFG -------------------------------------------------------------------------------
    dfg, start_act, end_act = pm4py.discover_dfg(df)
    p = os.path.join(OUT_DIR, f"{tag}_dfg_full.png")
    pm4py.save_vis_dfg(dfg, start_act, end_act, p)
    show(p)
    cut = DFG_MIN_EDGE_FRACTION * max(dfg.values())
    dfg_f = {e: c for e, c in dfg.items() if c >= cut}
    p = os.path.join(OUT_DIR, f"{tag}_dfg_filtered.png")
    pm4py.save_vis_dfg(dfg_f, start_act, end_act, p)
    show(p)
    try:
        perf, p_start, p_end = pm4py.discover_performance_dfg(df)
        perf_f = {e: v for e, v in perf.items() if e in dfg_f}
        p = os.path.join(OUT_DIR, f"{tag}_perfdfg.png")
        pm4py.save_vis_performance_dfg(perf_f, p_start, p_end, p)
        show(p)
    except Exception as exc:
        print(f"  [performance DFG skipped] {exc!r}")

    # Inductive Miner
    for nt in IM_NOISE_THRESHOLDS:
        net, im, fm = pm4py.discover_petri_net_inductive(df, noise_threshold=nt)
        p = os.path.join(OUT_DIR, f"{tag}_IM_noise{nt}_petri.png")
        pm4py.save_vis_petri_net(net, im, fm, p)
        show(p)
        try:
            bpmn = pm4py.discover_bpmn_inductive(df, noise_threshold=nt)
            p = os.path.join(OUT_DIR, f"{tag}_IM_noise{nt}_bpmn.png")
            pm4py.save_vis_bpmn(bpmn, p)
            show(p)
        except Exception as exc:
            print(f"  [BPMN skipped] {exc!r}")
        row = {"variant": tag, "miner": "IM", "parameter": nt, "places": len(net.places),
               "transitions": len(net.transitions), "arcs": len(net.arcs), **quality(df, net, im, fm)}
        results.append(row)
        print(f"  IM noise={nt}: {row}")

    # Heuristics Miner
    for dt in HM_DEPENDENCY_THRESHOLDS:
        try:
            heu = pm4py.discover_heuristics_net(df, dependency_threshold=dt)
            p = os.path.join(OUT_DIR, f"{tag}_HM_dep{dt}.png")
            pm4py.save_vis_heuristics_net(heu, p)
            show(p)
            net, im, fm = pm4py.discover_petri_net_heuristics(df, dependency_threshold=dt)
            row = {"variant": tag, "miner": "HM", "parameter": dt, "places": len(net.places),
                   "transitions": len(net.transitions), "arcs": len(net.arcs), **quality(df, net, im, fm)}
            results.append(row)
            print(f"  HM dependency={dt}: {row}")
        except Exception as exc:
            print(f"  [HM dependency={dt} skipped] {exc!r}")
    return results


# PIPELINE vs GROUND TRUTH
def edge_counts(df):
    """Directly-follows pairs (a, b) of a log -> Series of counts indexed by (a, b)."""
    d = df.sort_values(["case:concept:name", "time:timestamp"])
    nxt = d["concept:name"].shift(-1)
    same = d["case:concept:name"].eq(d["case:concept:name"].shift(-1))
    pairs = pd.DataFrame({"a": d["concept:name"][same].values, "b": nxt[same].values})
    return pairs.groupby(["a", "b"]).size()


def _relative(counts, index):
    return (counts / counts.sum()).reindex(index, fill_value=0.0)


def _cosine(x, y):
    den = np.sqrt((x ** 2).sum() * (y ** 2).sum())
    return float((x * y).sum() / den) if den > 0 else float("nan")


def activity_order(level, drop):
    acts = []
    for c in CLASSES:
        a = L1_MAP.get(c, c) if level == "L1" else c
        if a not in acts and not (drop and a == CARRYING):
            acts.append(a)
    return acts


def plot_dfg_comparison(rel_p, rel_g, level, drop, path, title_prefix="Directly-follows relation"):
    """Heatmaps of the directly-follows frequencies (GT, pipeline) and scatter of the two."""
    import matplotlib.pyplot as plt
    acts = activity_order(level, drop)
    n = len(acts)

    def matrix(rel):
        m = np.zeros((n, n))
        for (a, b), v in rel.items():
            if a in acts and b in acts:
                m[acts.index(a), acts.index(b)] = 100 * v
        return m

    mg, mp = matrix(rel_g), matrix(rel_p)
    vmax = max(mg.max(), mp.max(), 1e-9)
    fig, axes = plt.subplots(1, 3, figsize=(19, 6), gridspec_kw={"width_ratios": [1, 1, 0.9]})
    for ax, m, title in [(axes[0], mg, "Ground truth"), (axes[1], mp, "Pipeline")]:
        ax.imshow(m, cmap="Blues", vmin=0, vmax=vmax)
        ax.set_xticks(range(n)); ax.set_xticklabels(acts, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(n)); ax.set_yticklabels(acts, fontsize=8)
        ax.set_xlabel("next action"); ax.set_ylabel("action")
        ax.set_title(f"{title}: % of all transitions")
        for i in range(n):
            for j in range(n):
                if m[i, j] >= 0.5:
                    ax.text(j, i, f"{m[i, j]:.1f}", ha="center", va="center", fontsize=7,
                            color="white" if m[i, j] > 0.6 * vmax else "black")
    x, y = 100 * rel_g.values, 100 * rel_p.values
    ax = axes[2]
    ax.scatter(x, y, s=18, alpha=0.7)
    lim = max(x.max(), y.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=0.8)
    for k in np.argsort(-np.maximum(x, y))[:6]:
        a, b = rel_g.index[k]
        ax.annotate(f"{a[:8]}→{b[:8]}", (x[k], y[k]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("ground truth (% of transitions)"); ax.set_ylabel("pipeline (% of transitions)")
    ax.set_title("Edge frequencies")
    fig.suptitle(f"{title_prefix}, {level}{', without Carrying' if drop else ''}")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def dfg_comparison(logs):
    rows = []
    for level in LEVELS:
        for drop in DROP_CARRYING:
            df_p, df_g = logs[("pipe", level, drop)], logs[("gt", level, drop)]
            cp, cg = edge_counts(df_p), edge_counts(df_g)
            idx = cp.index.union(cg.index)
            p, g = _relative(cp, idx), _relative(cg, idx)
            top_p, top_g = set(p.nlargest(TOP_K_EDGES).index), set(g.nlargest(TOP_K_EDGES).index)
            acts = sorted(set(df_p["concept:name"]) | set(df_g["concept:name"]))
            ap = df_p["concept:name"].value_counts(normalize=True).reindex(acts, fill_value=0.0)
            ag = df_g["concept:name"].value_counts(normalize=True).reindex(acts, fill_value=0.0)

            ep_p = {episode_key(c): edge_counts(x) for c, x in df_p.groupby("case:concept:name")}
            ep_g = {episode_key(c): edge_counts(x) for c, x in df_g.groupby("case:concept:name")}
            common = sorted(set(ep_p) & set(ep_g), key=episode_order)
            matched, mismatched = [], []
            for e in common:
                for f in common:
                    i2 = ep_p[e].index.union(ep_g[f].index)
                    c = _cosine(_relative(ep_p[e], i2).values, _relative(ep_g[f], i2).values)
                    (matched if e == f else mismatched).append(c)
            row = {
                "level": level, "no_carrying": drop,
                "edges_gt": len(cg), "edges_pipeline": len(cp), "edges_common": len(set(cp.index) & set(cg.index)),
                "edge_pearson": round(float(p.corr(g)), 4), "edge_spearman": round(float(p.corr(g, method="spearman")), 4),
                "edge_cosine": round(_cosine(p.values, g.values), 4),
                f"top{TOP_K_EDGES}_jaccard": round(len(top_p & top_g) / len(top_p | top_g), 3),
                f"top{TOP_K_EDGES}_recall_of_gt": round(len(top_p & top_g) / len(top_g), 3),
                "activity_pearson": round(float(ap.corr(ag)), 4),
                "episode_cosine_matched_mean": round(float(np.mean(matched)), 4),
                "episode_cosine_matched_min": round(float(np.min(matched)), 4),
                "episode_cosine_mismatched_mean": round(float(np.mean(mismatched)), 4) if mismatched else float("nan"),
            }
            rows.append(row)
            print("  DFG comparison:", row)
            path = os.path.join(OUT_DIR, f"compare_dfg_{level}_{'noCarry' if drop else 'withCarry'}.png")
            plot_dfg_comparison(_relative(cp, idx), _relative(cg, idx), level, drop, path)
            show(path)
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT_DIR, "dfg_similarity.csv"), index=False)
    return out


# INTERVAL-AWARE COMPARISON
def interval_precedence_counts(events, level, drop_carrying):
    """Count COVER relations of the interval precedence partial order.

    Two events are ordered only if A.end <= B.start. A -> B is immediate
    when no third event C satisfies A.end <= C.start and C.end <= B.start.
    Overlapping events therefore are never artificially placed in sequence.
    This is NOT a PM4Py DFG and cannot be passed to standard Inductive Miner
    as though it were an ordinary trace log.
    """
    ev = events.copy()
    ev["action"] = ev["action"].replace(L1_MAP) if level == "L1" else ev["action"]
    if drop_carrying:
        ev = ev[ev["action"] != CARRYING]
    counts = Counter()
    for _, group in ev.groupby("recording"):
        if len(group) < 2:
            continue
        start = group["start"].astype("int64").to_numpy()
        end = group["end"].astype("int64").to_numpy()
        activities = group["action"].to_numpy()
        precedes = end[:, None] <= start[None, :]
        np.fill_diagonal(precedes, False)
        for i in range(len(group)):
            after_i = np.flatnonzero(precedes[i])
            if after_i.size == 0:
                continue
            indirect = precedes[np.ix_(after_i, after_i)].any(axis=0)
            for j in after_i[~indirect]:
                counts[(activities[i], activities[j])] += 1
    index = pd.MultiIndex.from_tuples(list(counts), names=["from", "to"])
    return pd.Series(list(counts.values()), index=index, dtype=float)


def concurrency_summary(events, source, level, drop_carrying):
    ev = events.copy()
    ev["action"] = ev["action"].replace(L1_MAP) if level == "L1" else ev["action"]
    if drop_carrying:
        ev = ev[ev["action"] != CARRYING]
    n_events, overlapping_events, tied_start_events, overlapping_pairs = 0, 0, 0, 0
    for _, group in ev.groupby("recording"):
        n = len(group)
        if n == 0:
            continue
        n_events += n
        start = group["start"].astype("int64").to_numpy()
        end = group["end"].astype("int64").to_numpy()
        overlaps = (start[:, None] < end[None, :]) & (start[None, :] < end[:, None])
        np.fill_diagonal(overlaps, False)
        overlapping_events += int(overlaps.any(axis=1).sum())
        overlapping_pairs += int(np.triu(overlaps, 1).sum())
        _, groupsizes = np.unique(start, return_counts=True)
        tied_start_events += int(groupsizes[groupsizes > 1].sum())
    return {
        "source": source, "level": level, "no_carrying": drop_carrying,
        "events": n_events, "events_with_overlaps": overlapping_events,
        "pct_events_with_overlaps": round(overlapping_events / n_events, 4) if n_events else None,
        "events_with_identical_start": tied_start_events,
        "pct_events_with_identical_start": round(tied_start_events / n_events, 4) if n_events else None,
        "overlapping_event_pairs": overlapping_pairs,
    }


def interval_precedence_comparison(pipe_events, gt_events):
    """Extra CSV and heatmap comparison that preserves overlap as unordered."""
    rows, conc_rows = [], []
    for level in LEVELS:
        for drop in DROP_CARRYING:
            for source, ev in (("pipeline", pipe_events), ("gt", gt_events)):
                conc_rows.append(concurrency_summary(ev, source, level, drop))
            cp = interval_precedence_counts(pipe_events, level, drop)
            cg = interval_precedence_counts(gt_events, level, drop)
            idx = cp.index.union(cg.index)
            p, g = _relative(cp, idx), _relative(cg, idx)
            cosine = _cosine(p.to_numpy(), g.to_numpy())
            pearson = float(p.corr(g)) if len(idx) >= 2 else float("nan")
            rows.append({
                "level": level, "no_carrying": drop,
                "edges_gt": len(cg), "edges_pipeline": len(cp),
                "edges_common": len(cg.index.intersection(cp.index)),
                "edge_cosine": round(cosine, 4), "edge_pearson": round(pearson, 4),
            })
            path = os.path.join(OUT_DIR, f"compare_interval_precedence_{level}_{'noCarry' if drop else 'withCarry'}.png")
            if len(idx):
                plot_dfg_comparison(p, g, level, drop, path,
                                    title_prefix="Immediate interval precedence (unordered overlaps)")
                show(path)
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "interval_precedence_similarity.csv"), index=False)
    pd.DataFrame(conc_rows).to_csv(os.path.join(OUT_DIR, "concurrency_statistics.csv"), index=False)
    return pd.DataFrame(rows)


def conformance_analysis(logs):
    """Models discovered from the GT (reference process); the pipeline log is replayed on them."""
    rows, per_trace = [], []
    for level in LEVELS:
        for drop in DROP_CARRYING:
            df_p, df_g = logs[("pipe", level, drop)], logs[("gt", level, drop)]
            for nt in IM_NOISE_THRESHOLDS:
                net, im, fm = pm4py.discover_petri_net_inductive(df_g, noise_threshold=nt)
                q_g, q_p = quality(df_g, net, im, fm), quality(df_p, net, im, fm)
                row = {"level": level, "no_carrying": drop, "gt_model_noise": nt,
                       **{f"gt_log_{k}": v for k, v in q_g.items()},
                       **{f"pipeline_log_{k}": v for k, v in q_p.items()}}
                rows.append(row)
                print("  Conformance:", row)
                if PER_TRACE_CONFORMANCE and COMPUTE_METRICS:
                    for case, sub in df_p.groupby("case:concept:name"):
                        try:
                            f = pm4py.fitness_token_based_replay(sub, net, im, fm)["log_fitness"]
                        except Exception:
                            f = float("nan")
                        per_trace.append({"level": level, "no_carrying": drop, "gt_model_noise": nt,
                                          "case": case, "trace_fitness": round(f, 4)})
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT_DIR, "conformance.csv"), index=False)
    if per_trace:
        pd.DataFrame(per_trace).to_csv(os.path.join(OUT_DIR, "conformance_per_trace.csv"), index=False)
    return out


def timing_analysis(event_tables):
    """event_tables: {'pipeline': events, 'gt': events}. Duration of the actions and events per minute."""
    stats = []
    for source, ev in event_tables.items():
        for level in LEVELS:
            d = ev.copy()
            d["activity"] = d["action"].replace(L1_MAP) if level == "L1" else d["action"]
            d["dur"] = (d["end"] - d["start"]).dt.total_seconds()
            g = d.groupby("activity")["dur"].agg(n="count", mean="mean", median="median", total="sum")
            g["p25"] = d.groupby("activity")["dur"].quantile(0.25)
            g["p75"] = d.groupby("activity")["dur"].quantile(0.75)
            g["share_of_action_time_pct"] = 100 * g["total"] / g["total"].sum()
            g = g.round(2).reset_index()
            g.insert(0, "level", level)
            g.insert(0, "source", source)
            stats.append(g)
    durations = pd.concat(stats)
    durations.to_csv(os.path.join(OUT_DIR, "action_durations.csv"), index=False)
    if len(event_tables) == 2:
        wide = durations.pivot_table(index=["level", "activity"], columns="source",
                                     values=["n", "mean", "share_of_action_time_pct"])
        wide.columns = [f"{a}_{b}" for a, b in wide.columns]
        wide["mean_ratio_pipeline_over_gt"] = (wide["mean_pipeline"] / wide["mean_gt"]).round(2)
        print("\nAction durations (seconds), pipeline vs ground truth:\n", wide.round(2).to_string())

    rows = []
    for source, ev in event_tables.items():
        for episode, x in ev.groupby("episode"):
            span = (x["end"].max() - x["start"].min()).total_seconds()
            rows.append({"source": source, "episode": episode, "span_s": round(span, 1), "events": len(x),
                         "events_per_min": round(60 * len(x) / span, 1) if span > 0 else float("nan")})
    timing = pd.DataFrame(rows)
    timing.to_csv(os.path.join(OUT_DIR, "case_timing.csv"), index=False)
    print("\nTiming per episode (mean over episodes):\n", timing.groupby("source")[["span_s", "events", "events_per_min"]]
          .mean().round(1).to_string())
    return durations, timing


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    events_all = load_event_table(OCEL_FILE)
    print(f"Cameras found: {sorted(events_all['camera'].unique())}")

    gt_events, gt_window = None, {}
    if USE_GT and CASE_NOTION != "recording":
        print("Ground-truth log skipped: it only exists with CASE_NOTION = 'recording'.")
    elif USE_GT:
        gt_paths = sorted(glob.glob(os.path.join(GT_DIR, "**", GT_FILE_GLOB), recursive=True),
                          key=lambda p: (gt_file_index(p), p))
        if not gt_paths:
            raise SystemExit(f"No {GT_FILE_GLOB} in {GT_DIR}")
        idx = [gt_file_index(p) for p in gt_paths]
        if len(set(idx)) != len(idx):
            raise SystemExit("Two label files have the same index (e.g. two 'labels.csv' in different "
                             "folders): the order of the files is ambiguous.")
        gt_data = {p: read_gt_file(p) for p in gt_paths}
        unknown = pd.concat([d[3] for d in gt_data.values()]).groupby(level=0).sum()
        if len(unknown):
            print(f"GT labels outside the {len(CLASSES)} classes (ignored): {unknown.to_dict()}")
        mapping = map_gt_files(events_all, gt_data)
        gt_events = build_gt_events(mapping, gt_data)
        gt_window = {e: (REFERENCE + pd.to_timedelta(gt_data[p][4] / GT_FPS, unit="s"),
                         REFERENCE + pd.to_timedelta((gt_data[p][2] + 1) / GT_FPS, unit="s"))
                     for e, p in mapping.items()}

    events = events_all[events_all["camera"] == CAMERA] if CAMERA else events_all
    if events.empty:
        raise SystemExit(f"No clip for camera {CAMERA}")
    if KEEP_MAIN_PERSON_ONLY:
        events = select_main_person(events)
    if gt_events is not None and RESTRICT_PIPELINE_TO_GT_EPISODES:
        events = events[events["episode"].isin(gt_events["episode"].unique())]
    if gt_events is not None and RESTRICT_PIPELINE_TO_GT_SPAN:
        n_before = len(events)
        w_start = events["episode"].map(lambda e: gt_window[e][0])
        w_end = events["episode"].map(lambda e: gt_window[e][1])
        events = events[(events["end"] > w_start) & (events["start"] < w_end)].copy()
        clipped_starts = events["episode"].map(lambda e: gt_window[e][0])
        clipped_ends = events["episode"].map(lambda e: gt_window[e][1])
        events["start"] = events["start"].where(events["start"] >= clipped_starts, clipped_starts)
        events["end"] = events["end"].where(events["end"] <= clipped_ends, clipped_ends)
        events = events[events["start"] < events["end"]].copy()
        print(f"Pipeline events outside the GT interval: {n_before - len(events)} of {n_before} removed "
              "(boundary-crossing events clipped)")
    if MERGE_CLIP_BOUNDARIES:
        n_before_merge = len(events)
        events, boundary_audit = merge_cross_clip_fragments(events, BOUNDARY_TOLERANCE_S)
        boundary_audit.to_csv(os.path.join(OUT_DIR, "boundary_merges_applied.csv"), index=False)
        print(f"Cross-clip merges: {len(boundary_audit)}; pipeline events: "
              f"{n_before_merge} -> {len(events)}")
        if not boundary_audit.empty:
            print("Merges by action:\n", boundary_audit["action"].value_counts().to_string())

    print(f"Pipeline: {len(events)} events, {events['recording'].nunique()} recordings, "
          f"{events['clip'].nunique()} clips")
    sources = [("pipe", f"pipe_{CAMERA or 'allcams'}_{CASE_NOTION}", events, CASE_NOTION)]
    if gt_events is not None:
        print(f"Ground truth: {len(gt_events)} events, {gt_events['episode'].nunique()} episodes")
        sources.append(("gt", "gt_recording", gt_events, "recording"))

    all_metrics, all_stats, logs = [], [], {}
    for kind, source, ev, notion in sources:
        for level in LEVELS:
            for drop in DROP_CARRYING:
                tag = f"{source}_{level}_{'noCarry' if drop else 'withCarry'}"
                print(f"\n=== {tag} ===")
                df = build_log(ev, level, drop, notion)
                logs[(kind, level, drop)] = df
                stats = {"variant": tag, **log_stats(df)}
                print("  log:", stats)
                all_stats.append(stats)
                all_metrics += run_variant(df, tag)

    pd.DataFrame(all_stats).to_csv(os.path.join(OUT_DIR, "log_statistics.csv"), index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(os.path.join(OUT_DIR, "model_metrics.csv"), index=False)
    print("\n", metrics.to_string(index=False))

    # pipeline vs ground truth, and time perspective
    if gt_events is not None:
        print("\n=== DFG comparison (pipeline vs ground truth) ===")
        print(dfg_comparison(logs).to_string(index=False))
        print("\n=== Supplemental interval-aware precedence (overlaps unordered) ===")
        print(interval_precedence_comparison(events, gt_events).to_string(index=False))
        print("\n=== Conformance checking: pipeline log replayed on the models discovered from the GT ===")
        print(conformance_analysis(logs).to_string(index=False))
        timing_analysis({"pipeline": events, "gt": gt_events})
    else:
        timing_analysis({"pipeline": events})
    print(f"\nFigures and CSVs saved in {OUT_DIR}")


if __name__ == "__main__":
    main()